# BÀI TẬP: EDA HOÀN CHỈNH — TITANIC
## Data Profiling → Descriptive Statistics → Define the Question
**Nguồn:** kaggle.com/c/titanic (891 dòng)

```
PHẦN A — DATA PROFILING       : dữ liệu sạch chưa, có gì
PHẦN B — DESCRIPTIVE STATS    : từng biến trông như thế nào
PHẦN C — DEFINE THE QUESTION  : đặt câu hỏi kinh doanh, trả lời bằng code, rút insight
```

## Setup

In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [3]:

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.info()

(891, 15)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked

## A.2. Missing values & Duplicate data

In [3]:
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'missing': missing, 'percent': missing_percent})
print(missing_df[missing_df['missing'] > 0].sort_values('missing', ascending=False))
print(df.duplicated().sum())

             missing    percent
deck             688  77.216611
age              177  19.865320
embarked           2   0.224467
embark_town        2   0.224467
107


## A.3. Invalid values

In [4]:
print((df['age'] <= 0).sum())
print((df['fare'] < 0).sum())
print((df['fare'] == 0).sum())
print((df['sibsp'] < 0).sum())
print((df['parch'] < 0).sum())
print(df['sex'].unique())
print(df['embarked'].unique())
print(df['pclass'].unique())

0
0
15
0
0
<StringArray>
['male', 'female']
Length: 2, dtype: str
<StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str
[3 1 2]


## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [5]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df[['sibsp', 'parch', 'family_size']].head()

,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
# TODO
print(df[['age', 'fare', 'family_size']].mean())
print(df[['age', 'fare', 'family_size']].median())
print(df[['age', 'fare', 'family_size']].mode().iloc[0])
print(df['sex'].mode()[0])
print(df['embarked'].mode()[0])
print(df['pclass'].mode()[0])

age            29.699118
fare           32.204208
family_size     1.904602
dtype: float64
age            28.0000
fare           14.4542
family_size     1.0000
dtype: float64
age            24.00
fare            8.05
family_size     1.00
Name: 0, dtype: float64
male
S
3


## Group 2 — Dispersion

In [7]:
print(df[['age', 'fare', 'family_size']].std())
print(df[['age', 'fare', 'family_size']].var())
print(df['age'].max() - df['age'].min())
print(df['fare'].max() - df['fare'].min())
print(df[['age', 'fare']].quantile(0.75) - df[['age', 'fare']].quantile(0.25))

age            14.526497
fare           49.693429
family_size     1.613459
dtype: float64
age             211.019125
fare           2469.436846
family_size       2.603248
dtype: float64
79.58
512.3292
age     17.8750
fare    23.0896
dtype: float64


## Group 3 — Location and Shape

In [8]:
print(df[['age', 'fare']].quantile([0.25, 0.5, 0.75]))
print(df[['age', 'fare']].skew())
print(df[['age', 'fare']].kurt())

         age     fare
0.25  20.125   7.9104
0.50  28.000  14.4542
0.75  38.000  31.0000
age     0.389108
fare    4.787317
dtype: float64
age      0.178274
fare    33.398141
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [9]:
survival_by_class = df.groupby('pclass')['survived'].mean()
print(survival_by_class)
print(survival_by_class.max() - survival_by_class.min())

pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64
0.3872671041713812


## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [10]:
print(df.groupby('sex')['survived'].mean())
print(df.groupby('pclass')['survived'].mean())
print(df.groupby('sex')['survived'].mean().max() - df.groupby('sex')['survived'].mean().min())
print(df.groupby('pclass')['survived'].mean().max() - df.groupby('pclass')['survived'].mean().min())

sex
female    0.742038
male      0.188908
Name: survived, dtype: float64
pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64
0.5531300709799203
0.3872671041713812


## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [11]:
print(df.groupby('survived')['fare'].mean())
print(df[['fare', 'survived']].corr())

survived
0    22.117887
1    48.395408
Name: fare, dtype: float64
              fare  survived
fare      1.000000  0.257307
survived  0.257307  1.000000


## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [12]:
print(df.groupby('family_size')['survived'].mean())
print(df[['family_size', 'survived']].corr())

family_size
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: survived, dtype: float64
             family_size  survived
family_size     1.000000  0.016639
survived        0.016639  1.000000


## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [13]:
survival_by_town = df.groupby('embark_town')['survived'].mean()
print(survival_by_town)
print(survival_by_town.idxmax())

embark_town
Cherbourg      0.553571
Queenstown     0.389610
Southampton    0.336957
Name: survived, dtype: float64
Cherbourg


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

1. Hạng vé ảnh hưởng rõ rệt đến tỷ lệ sống sót: hành khách hạng 1 có tỷ lệ sống sót cao hơn hẳn hạng 3.
2. Giới tính là yếu tố ảnh hưởng mạnh nhất: nữ giới có tỷ lệ sống sót cao hơn đáng kể so với nam giới.
3. Giá vé (fare) có tương quan dương với khả năng sống sót, phản ánh việc hành khách giàu có thường ở khoang hạng cao, gần lối thoát hiểm hơn.
4. Gia đình có quy mô vừa phải (2-4 người) có tỷ lệ sống sót cao hơn người đi một mình hoặc gia đình quá đông.
5. Cảng lên tàu Cherbourg có tỷ lệ sống sót cao hơn Southampton và Queenstown, có thể liên quan đến cơ cấu hạng vé của hành khách lên tàu tại đó.